# Pokémon Gen 2 OU — Candidate 7010

This notebook clones the project's **maintained** GitHub pipeline and reproduces its Kaggle-ready predictions using organizer data only. The 70.10% value is exploratory five-fold OOF accuracy, not a Kaggle leaderboard result.

## 1. Clone the source and install pinned dependencies

Kaggle Internet access may be necessary to clone GitHub. It is not used as an input to any model.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/fds-pokemon")
REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise ValueError(f"Not a Git repository: {REPO_DIR}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Repository commit:", COMMIT)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_DIR / "requirements.txt"),
], check=True)

## 2. Locate the organizer's competition files

In [ ]:
train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected one train.jsonl; found {len(train_files)}")

DATA_DIR = train_files[0].parent
for name in ("test.jsonl", "type_chart_gen2.csv", "sample_submission.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
print("Official data:", DATA_DIR)

## 3. Reproduce the selected candidate submission

This runs the full 7010 ensemble, verifies the official submission schema, and does not contact Kaggle's scoring system.

In [ ]:
OUTPUT = Path("/kaggle/working/submission_candidate_7010.csv")
subprocess.run([
    sys.executable, str(REPO_DIR / "run.py"),
    "--data", str(DATA_DIR),
    "--output", str(OUTPUT),
], check=True)

import pandas as pd
prediction = pd.read_csv(OUTPUT)
print(f"Created {len(prediction)} predictions:", OUTPUT)
display(prediction.head())

## Final submission selection

The competition requires three selected submissions and a public notebook reproducing them. Additional selected versions should be documented here after the final selection. The final methodological report is a separate deliverable.